In [1]:
# !pip install transformers

from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# Load GPT-2 (117M) model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Let's make sure that model uses GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def generoi_teksti(alku, max_pituus=100, temperature=0.7, top_p=0.9): # MODIFIED THE FUNCTION TO ACCEPT TEMPERATURE AND TOP_P AS PARAMETERS FOR TESTING
    # Tokenization
    input_ids = tokenizer.encode(alku, return_tensors="pt")
    attention_mask = torch.ones_like(input_ids)  # Kaikki tokenit ovat merkityksellisiä

    # transfer to GPU (or to CPU)
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    # Text generation
    output = model.generate(
        input_ids,
        do_sample=True,
        attention_mask=attention_mask,
        max_length=max_pituus,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        temperature=temperature, # REMOVED HARD-CODED VALUE AND REPLACED WITH PARAMETER
        top_p=top_p, # DID THE SAME WITH THIS PARAMETER
        pad_token_id=tokenizer.eos_token_id  # to remove warning
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

c:\Users\avins\OneDrive - Oulun ammattikorkeakoulu\Työpöytä\Week 3\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4128.83it/s]


In [3]:
# Example
alkuteksti = "Suomi on kaunis maa, jossa"
generoitu_teksti = generoi_teksti(alkuteksti, max_pituus=50)
print(generoitu_teksti)

Suomi on kaunis maa, jossa bijan (I am sure that will be fine)

The name of the song is "Bijani" (In Japanese) and it's a song by the band.


In [2]:
# STEP 1: Compare how model works with Finnish and English languages. ##
english_prompt = "Finland is a beautiful country, where"
finnish_prompt = "Suomi on kaunis maa, jossa"

print("--- ENGLISH OUTPUT ---")
print(generoi_teksti(english_prompt, max_pituus=50))

print("\n--- FINNISH OUTPUT ---")
print(generoi_teksti(finnish_prompt, max_pituus=50))

--- ENGLISH OUTPUT ---
Finland is a beautiful country, where there is no shortage of sunshine, and in the winter, when it rains, there are many beautiful beaches.

But even at the time of our arrival in Spain, the beach is not quite so pleasant

--- FINNISH OUTPUT ---
Suomi on kaunis maa, jossa on moume, kimme on uma, kevme bien.

Mister, I will not be silent on your side. I don't care who you


Answer after **4.3 seconds**:
```
--- ENGLISH OUTPUT ---
Finland is a beautiful country, where people live in a different way. They live on a continent, and you can't really do that in Scandinavia. There's a lot of things that are very different.

The idea of a Scandinavian

--- FINNISH OUTPUT ---
Suomi on kaunis maa, jossa hao.

Kee-haw, nae-waa-saa. (If I've got a bad smell, I'll kill it.)
 (I'm not
```

### English output
Grammatically looks correct, but the context of the text is quite funny. Stating that people "_live on a continent, and you can't really do that in Scandinavia._"  
We're guessing this model doesn't understand geography since it's trained to just find the next word based on math and propability and not by the context on its own. 

### Finnish output
Also quite funny, but this time does not make any sense directly after the hardcoded prompt. Then switching to English with no relation to Finland whatsoever and kind of nonsense "_(If I've got a bad smell, I'll kill it.)
 (I'm not_"  
Here we're guessing the model just went completely in on itself, as it must have 0 training with Finnish. So it immediately goes into nonsense words and then switching into English because that's the training data.


In [3]:
## STEP 2: Prove that model hallucinates. ##

hallucination_prompt = "LEGO was a company founded in 1932 by"

print(generoi_teksti(hallucination_prompt, max_pituus=60))

LEGO was a company founded in 1932 by the late Jules Verne and William H. McBride, who worked as a vice president and managing director of the New York City office of New Line. The company was incorporated in 1964.

The company's name was originally a reference to the


Answer after **3.0 seconds**:

```
LEGO was a company founded in 1932 by the young inventor Charles Dickens, who had become a lifelong fan of the comic books.

It was then that the company began its own series of stories, in which the characters were created by cartoonists. The comic book industry began to boom, and
```

### Hallucination
It's pretty clear that the generated text is grammatically correct, but the context is completely lost and shows that is it hallucinating.  
**Charles Dickens** lived from 1812-1870 so founding a Danish company in 1932 seems like a stretch. It then goes into explaining about comic books and the industry which has nothing to do with LEGO. 

In [4]:
## STEP 3: Test effect of temperature parameter. ##

test_prompt = "Once upon a time, in a land far away, there lived a"

# LOW TEMPERATURE WILL PICK MORE PROBABLE WORDS, RESULTING IN MORE COHERENT BUT LESS DIVERSE OUTPUT
print("----- TEMPERATURE 0.1 -----")
print(generoi_teksti(test_prompt, temperature=0.1, max_pituus=50))

# HIGHER TEMPERATURE WILL PICK LESS PROBABLE WORDS, RESULTING IN MORE DIVERSE BUT LESS COHERENT OUTPUT
print("\n----- TEMPERATURE 1.5 -----")
print(generoi_teksti(test_prompt, temperature=1.5, max_pituus=50))

----- TEMPERATURE 0.1 -----
Once upon a time, in a land far away, there lived a man who had been born in the land of the dead. He was a son of a rich man, and he had a wife who was of noble birth. The man was called

----- TEMPERATURE 1.5 -----
Once upon a time, in a land far away, there lived a righteous race; their blood lay long before it was made; from which they sought, from that which their fathers made, every man shall possess that holy place."

It was


Answer after **4.3 seconds**:

```
----- TEMPERATURE 0.1 -----
Once upon a time, in a land far away, there lived a man who had been born in the land of the dead. He was a son of a great man, and he was born of an old man. And he had a wife,

----- TEMPERATURE 1.5 -----
Once upon a time, in a land far away, there lived a prophet; there had ever been prophet. There had come unto them the Lord Jesus of Nazareth who, while his life was beginning, the God they saw in Jerusalem had betrayed him
```

### Low temperature
This is where basically the most propable word will be chosen next every time. So after the prompt it directly goes into writing about a man, born, son, wife. Extremely safe and generic.

### High temperature
Unlikely words have a higher chance of being picked, and honestly goes into a complete rant. The prompt started as a fairy tale and this temperature ends up being about prophets and betrayal in Jerusalem. 

In [5]:
## STEP 4: Test effect of top_p parameter ##

test_prompt = "When a child grows up, they often dream of"

# LOWER TOP_P WILL CONSIDER ONLY THE MOST PROBABLE WORDS, RESULTING IN MORE COHERENT BUT LESS DIVERSE OUTPUT
print("----- TOP_P 0.1 -----")
print(generoi_teksti(test_prompt, top_p=0.1, max_pituus=50))

# HIGHER TOP_P WILL CONSIDER A WIDER RANGE OF WORDS, RESULTING IN MORE DIVERSE BUT LESS COHERENT OUTPUT
print("\n----- TOP_P 0.9 -----")
print(generoi_teksti(test_prompt, top_p=0.9, max_pituus=50))

----- TOP_P 0.1 -----
When a child grows up, they often dream of being able to play with their parents.

"I think it's a very important thing for children to be able, to have a sense of belonging and to feel safe," said Dr. David

----- TOP_P 0.9 -----
When a child grows up, they often dream of being able to play with their parents and to learn from their experiences. They may not realize how important it is to have children and how much they need to be loved. However, the child is not


Answer after **5.4 seconds**:

```
----- TOP_P 0.1 -----
When a child grows up, they often dream of being able to play with their parents.

"I think it's a very important thing for children to be able, to have a sense of belonging and to feel safe," said Dr. Michael

----- TOP_P 0.9 -----
When a child grows up, they often dream of being born into the same world, and they are given the opportunity to explore that world. I think that's the most important thing about this book, because I believe that there are certain things that we
```

### why top_p
When the top_p is set to 0,1 it will grade the propabilities for each token, and sort so only the top tokens which propability sums up to 10 % and be able to be picked from, limiting the options a lot. On the other hand a top_p of 90% means the tokens it can choose from only sorts out the bottom 10 % which are most likely when the model will start saying complete nonsense that wouldn't make sense with the context at all.

In [21]:
## STEP 5: Test models in-context-learning-capability (few-shot prompt) ##

# FEW-SHOT PROMPT PROVIDES EXAMPLES OF THE TASK TO THE MODEL, HELPING IT TO UNDERSTAND WHAT KIND OF OUTPUT IS EXPECTED. 
few_shot_prompt = """
Object: Grass -> Color: Green
Object: Sky -> Color: Blue
Object: Strawberry -> Color: Red
Object: Banana -> Color:"""

print(generoi_teksti(few_shot_prompt, max_pituus=33))


Object: Grass -> Color: Green
Object: Sky -> Color: Blue
Object: Strawberry -> Color: Red
Object: Banana -> Color: Black



Answer after **0.1 seconds**:

```
Object: Grass -> Color: Green
Object: Sky -> Color: Blue
Object: Strawberry -> Color: Red
Object: Banana -> Color: Yellow

```
Running it many times we can sometimes get different possibilities like, black and orange. These options can occur for a banana, like an overwipe banana being brown or almost black but the majority of results end on yellow.

Because the model has `do_sample=True` it **will** sometimes pick options that does not necessarily have the highest propability.  


## Step 4 Tiktokeniser exercise
```
<|im_start|>user<|im_sep|>How far is moon?<|im_end|><|im_start|>assistant<|im_sep|>very far<|im_end|><|im_start|>user<|im_sep|>How far is sun?<|im_end|><|im_start|>assistant<|im_sep|>

200264, 1428, 200266, 5299, 4150, 382, 28479, 30, 200265, 200264, 173781, 200266, 1437, 4150, 200265, 200264, 1428, 200266, 5299, 4150, 382, 7334, 30, 200265, 200264, 173781, 200266

Explain this token sequence!
```



The given sequence of integers represents a tokenized form of text used as input to a language model. In natural language processing, raw text is first broken down into smaller units called tokens. These tokens may correspond to whole words, subwords, or individual characters. Each token is then mapped to a unique integer ID based on the model’s vocabulary.

In this sequence, smaller numerical values correspond to tokens representing actual textual content, such as words or punctuation. Larger values (for example, numbers above 200000) are typically reserved for special tokens. These special tokens are used to encode structural information, such as the beginning or end of a sequence, or to distinguish between different roles in a conversation (e.g., user and assistant).

The entire sequence therefore encodes both the content and structure of a dialogue in a numerical format. This representation allows the model to efficiently process and generate language, as it operates on numerical data rather than raw text.